In [1]:
import json
from collections import defaultdict

with open("../../../data/processed/book4/done/adaptive_chunks.json", "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

with open("../../../data/processed/book4/done/final_chunk_ready.json", "r", encoding="utf-8") as f:
    sections_data = json.load(f)

In [2]:
section_lookup = {s["section_id"]: s for s in sections_data}
documents = {}
for s in sections_data:
    doc_id = s["doc_id"]
    if doc_id not in documents:
        documents[doc_id] = {
            "doc_id": doc_id,
            "doc_version": s.get("doc_version"),
            "doc_date": s.get("doc_date")
        }

documents_list = list(documents.values())

# ---------- Extract sections ----------
sections_list = []
blocks_list = []
tables_list = []

for s in sections_data:
    sections_list.append({
        "section_id": s["section_id"],
        "doc_id": s["doc_id"],
        "parent_section_id": s.get("parent_section_id"),
        "title": s.get("title"),
        "section_number": s.get("section_number"),
        "level": s.get("level"),
        "parent_titles": s.get("parent_titles"),
        "start_page": s.get("start_page"),
        "end_page": s.get("end_page"),
#        "start_top": s.get("start_top"),
#        "end_top": s.get("end_top"),
        "child_section_ids": s.get("child_section_ids")
    })

    # ---------- Blocks ----------
    for b in s.get("blocks", []):
        blocks_list.append({
            "block_id": b["block_id"],
            "section_id": s["section_id"],
            "doc_id": s["doc_id"],
            "type": b.get("type"),
            "page_num": b.get("page_num"),
            "text": b.get("text"),
            # "bbox_x0": b.get("bbox.x0"),
            # "bbox_top": b.get("bbox.top"),
            # "bbox_x1": b.get("bbox.x1"),
            # "bbox_bottom": b.get("bbox.bottom"),
            "items": b.get("items"),
            "intro_block_id": b.get("intro_block_id")
        })

    # ---------- Tables ----------
    for t in s.get("tables", []):
        tables_list.append({
            "table_id": t["table_id"],
            "section_id": s["section_id"],
            "doc_id": t.get("doc_id"),
            "doc_version": t.get("doc_version"),
            "doc_date": t.get("doc_date"),
            "page_num": t.get("page_num"),
            "bbox": t.get("bbox"),
            "title": t.get("title"),
            "structured_data": t.get("structured_data"),
            "table_text": t.get("table_text")
        })

In [3]:
chroma_ready = []
chunks_list = []
chunk_block_links = []
chunk_merge_links = []

for c in chunks_data:

    section = section_lookup.get(c["section_id"], {})
    doc_version = section.get("doc_version")
    doc_date = section.get("doc_date")

    # ---------- CHROMA FORMAT ----------
    chroma_ready.append({
        "id": c["chunk_id"],
        "document": c["text"],
        "metadata": {
            "doc_id": c["doc_id"],
            "doc_version": doc_version,
            "doc_date": doc_date,
            "section_id": c["section_id"],
            "parent_section_id": c.get("parent_section_id"),
            "title": c.get("title"),
            "section_number": c.get("section_number"),
            "level": c.get("level"),
            "parent_titles": " > ".join(c.get("parent_titles", [])) if isinstance(c.get("parent_titles"), list) else c.get("parent_titles"),
            "chunk_index": c.get("chunk_index"),
            "page_num": c.get("page_num"),
            "type": c.get("type"),
            "table_id": c.get("table_id"),
            "table_title": c.get("table_title"),
            "token_count": c.get("token_count"),
            "candidate_source": c.get("candidate_source"),
            "chunking_reason": c.get("chunking_reason"),
            "size_band": c.get("size_band"),
            "is_split": c.get("is_split"),
            "split_group_id": c.get("split_group_id"),
            "merge_group_size": c.get("merge_group_size")
        }
    })

    # ---------- POSTGRES CHUNKS ----------
    chunks_list.append({
        "chunk_id": c["chunk_id"],
        "doc_id": c["doc_id"],
        "section_id": c["section_id"],
        "chunk_index": c.get("chunk_index"),
        "title": c.get("title"),
        "section_number": c.get("section_number"),
        "level": c.get("level"),
        "parent_titles": c.get("parent_titles"),
        "parent_section_id": c.get("parent_section_id"),
        "text": c.get("text"),
        "token_count": c.get("token_count"),
        "type": c.get("type"),
        "block_count": c.get("block_count"),
        "is_split": c.get("is_split"),
        "split_group_id": c.get("split_group_id"),
        "candidate_source": c.get("candidate_source"),
        "chunking_reason": c.get("chunking_reason"),
        "size_band": c.get("size_band"),
        "oversize_reason": c.get("oversize_reason"),
        "merge_group_size": c.get("merge_group_size"),
        "page_num": c.get("page_num"),
        "page_span": c.get("page_span"),
        "table_id": c.get("table_id"),
        "table_title": c.get("table_title"),
        "overlap_prev_tokens": c.get("overlap_prev_tokens"),
    })

    # ---------- Chunk ↔ Block links ----------
    for i, block_id in enumerate(c.get("source_block_ids", [])):
        chunk_block_links.append({
            "chunk_id": c["chunk_id"],
            "block_id": block_id,
            "ordinal": i
        })

    # ---------- Merge lineage ----------
    merged_chunks = c.get("merged_from_chunk_ids", [])
    merged_sections = c.get("merged_from_section_ids", [])

    for i, mc in enumerate(merged_chunks):
        chunk_merge_links.append({
            "chunk_id": c["chunk_id"],
            "merged_from_chunk_id": mc,
            "merged_from_section_id": merged_sections[i] if i < len(merged_sections) else None,
            "ordinal": i
        })


In [4]:

# ---------- Save Chroma file ----------
with open("../../../data/processed/book4/done/chroma_ready.json", "w", encoding="utf-8") as f:
    json.dump(chroma_ready, f, indent=2, ensure_ascii=False)

# ---------- Save Postgres file ----------
postgres_ready = {
    "documents": documents_list,
    "sections": sections_list,
    "blocks": blocks_list,
    "tables": tables_list,
    "chunks": chunks_list,
    "chunk_block_links": chunk_block_links,
    "chunk_merge_links": chunk_merge_links
}

with open("../../../data/processed/book4/done/postgres_ready.json", "w", encoding="utf-8") as f:
    json.dump(postgres_ready, f, indent=2, ensure_ascii=False)

print("Files created:")
print(" - chroma_ready.json")
print(" - postgres_ready.json")

Files created:
 - chroma_ready.json
 - postgres_ready.json
